# 04 — Self-Supervised Napkin Sketch → Real BIM LoRA (`google/gemma-4-31B-it`)

Zero manual labeling: this notebook generates hundreds of **parametric MutinyBIM floor plans** across 6 architectural topologies, verifies every script against live `@permitzip/bim-script` + `bim-lint`, renders organic **hand-drawn napkin / graph-paper / iPad / phone-photo sketches**, and trains a multimodal **`ΔW_sketch_to_bim` (`vision_floorplan`) LoRA**!

### Why Weight-Space Composability Still Works Here
We **freeze Gemma 4's vision tower** (`finetune_vision_layers=False`) and train the LoRA **strictly on the `spec.json` language-backbone modules** (`q/k/v/o_proj`, `gate/up/down_proj`, `r=16`, `lora_alpha=16`). That means your trained `ΔW_sketch_to_bim` adapter lives in the **exact same weight space** as `ΔW_mechanical`, `ΔW_electrical`, and `-λ·ΔW_clash` — ready to blend in Notebook `02` and Studio `03`!

### Hardware & Credit Budget
* **Recommended GPU:** **A100 (40GB or 80GB)** (~7–9 CCUs/hr; ~20–40 CCUs for a full 120-sample run).


In [ ]:
#@title 1. Check GPU, Clone Repos, Install Node 22, and Boot Warm MutinyBIM Verifier
!nvidia-smi

import json, subprocess, sys, time
from pathlib import Path

MIXER_REPO_URL = "https://github.com/5hyycjb77y/mutinybim-mixer.git" #@param {type:"string"}
WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
%cd /content/work

if not (WORK_DIR / "mutinybim-mixer").exists():
    !git clone --depth 1 {MIXER_REPO_URL} mutinybim-mixer

MIXER_DIR = WORK_DIR / "mutinybim-mixer"
SPEC = json.loads((MIXER_DIR / "spec.json").read_text())
MUTINY_DIR = WORK_DIR / "mutinybim"

if not MUTINY_DIR.exists():
    !git clone {SPEC["mutinybim"]["repo"]} {MUTINY_DIR}
!git -C {MUTINY_DIR} fetch --depth 1 origin {SPEC["mutinybim"]["commit"]} && git -C {MUTINY_DIR} checkout {SPEC["mutinybim"]["commit"]}

node_v = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_v.returncode != 0 or int(node_v.stdout.strip().lstrip("v").split(".")[0]) < 22:
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs
!node -v

if not (MUTINY_DIR / "node_modules").exists():
    !npm --prefix {MUTINY_DIR} install --no-audit --no-fund

verifier_log = open("/content/verifier.log", "w")
verifier_proc = subprocess.Popen(
    ["node", str(MIXER_DIR / "verifier/server.mjs"), "--mutinybim", str(MUTINY_DIR), "--port", "8787"],
    stdout=verifier_log,
    stderr=subprocess.STDOUT,
)

sys.path.insert(0, str(MIXER_DIR))
from verifier.verify_client import MutinyVerifierClient

client = MutinyVerifierClient("http://127.0.0.1:8787", timeout=60.0)
for _ in range(30):
    try:
        print("MutinyBIM Verifier ready:", client.health())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Verifier failed to start — check /content/verifier.log")


In [ ]:
#@title 2. Install Multimodal Fine-Tuning & Sketch Dependencies
!pip install -q --upgrade pip
!pip install -q unsloth trl peft accelerate bitsandbytes safetensors datasets huggingface_hub pillow numpy matplotlib cairosvg


In [ ]:
#@title 3. Training & Dataset Configuration
NUM_TRAIN_SAMPLES = 120 #@param {type:"integer"}
NUM_EVAL_SAMPLES = 12 #@param {type:"integer"}
VERIFY_GROUND_TRUTH = True #@param {type:"boolean"}
INCLUDE_SVG_PHOTO_AUG = True #@param {type:"boolean"}
MAX_SFT_STEPS = 120 #@param {type:"integer"}
LEARNING_RATE = 2e-4 #@param {type:"number"}
ENABLE_THINKING = False #@param {type:"boolean"}
REAL_NAPKIN_DIR = "/content/real_napkins" #@param {type:"string"}
HF_OUTPUT_REPO = "5hyycjb77y/mutinybim-gemma4-31b-sketch-to-bim-v1" #@param {type:"string"}
PUSH_TO_HUB = False #@param {type:"boolean"}

OUT_DIR = Path("/content/outputs/sketch_to_bim")
DATA_DIR = OUT_DIR / "dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Configured {NUM_TRAIN_SAMPLES} train + {NUM_EVAL_SAMPLES} eval sketches | steps={MAX_SFT_STEPS} | repo={HF_OUTPUT_REPO}")


In [ ]:
#@title 4. Generate Self-Supervised Napkin Sketches + Verify Ground Truth + 3x3 Gallery
import matplotlib.pyplot as plt
from PIL import Image
from mixer.prompts import SYSTEM_PROMPT_TEMPLATE
from mixer.sketch_engine import (
    SKETCH_STYLES,
    generate_sketch_dataset_specs,
    render_parametric_sketch_png,
    sketchify_image,
)
from mixer.studio_render import svg_to_png

REF_MD = client.reference(compact=True)["markdown"]

train_specs = generate_sketch_dataset_specs(NUM_TRAIN_SAMPLES, start_seed=1000)
eval_specs = generate_sketch_dataset_specs(NUM_EVAL_SAMPLES, start_seed=9000)

def prepare_records(specs, split_name):
    records = []
    by_topo = {}
    if not VERIFY_GROUND_TRUTH:
        print(f"WARNING [{split_name}]: VERIFY_GROUND_TRUTH=False — admitting unverified scripts!")

    for idx, sample in enumerate(specs):
        stats = by_topo.setdefault(sample.topology, {"total": 0, "admitted": 0})
        stats["total"] += 1
        style = SKETCH_STYLES[idx % len(SKETCH_STYLES)]
        img_path = DATA_DIR / f"{split_name}_{sample.sample_id}.png"
        gt_svg = ""

        if VERIFY_GROUND_TRUTH:
            if INCLUDE_SVG_PHOTO_AUG and idx % 4 == 3:
                bundle = client.export_bundle(
                    sample.ts_source,
                    project_name="Sketch Dataset",
                    expected_kinds=sample.expected_kinds,
                    include_ifc=False,
                    include_pdf=False,
                )
                vr = bundle.verify
                admitted = vr.ok and vr.breakdown.get("spec_match", 0.0) == 1.0
                if not admitted:
                    continue
                if bundle.sheets:
                    gt_svg = bundle.sheets[0]["svg"]
                    raw_png = DATA_DIR / f"{split_name}_{sample.sample_id}_raw.png"
                    svg_to_png(gt_svg, raw_png, output_width=896)
                    sketchify_image(raw_png, img_path, style="phone_photo_blueprint", seed=sample.seed)
                else:
                    render_parametric_sketch_png(sample, img_path, style=style)
            else:
                vr = client.verify(sample.ts_source, expected_kinds=sample.expected_kinds)
                admitted = vr.ok and vr.breakdown.get("spec_match", 0.0) == 1.0
                if not admitted:
                    continue
                render_parametric_sketch_png(sample, img_path, style=style)

        stats["admitted"] += 1
        records.append({
            "sample": sample,
            "style": style,
            "image_path": str(img_path),
            "gt_svg": gt_svg,
        })

    print(f"[{split_name}] Admitted {len(records)}/{len(specs)} samples:")
    for topo_name, st in sorted(by_topo.items()):
        print(f"  - {topo_name}: {st['admitted']}/{st['total']}")
    if VERIFY_GROUND_TRUTH:
        empty_topos = [t for t, st in by_topo.items() if st["admitted"] == 0]
        if empty_topos:
            raise RuntimeError(f"[{split_name}] Topologies failed completely (0 admitted): {empty_topos}")
        if specs and (len(specs) - len(records)) / len(specs) > 0.05:
            raise RuntimeError(f"[{split_name}] Drop rate exceeded 5% ({len(records)}/{len(specs)} admitted)")
    return records

train_records = prepare_records(train_specs, "train")
eval_records = prepare_records(eval_specs, "eval")

# Plot a 3x3 visual gallery of generated napkin/graph/iPad sketches
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, rec in zip(axes.flat, train_records[:9]):
    ax.imshow(Image.open(rec["image_path"]))
    ax.set_title(f"{rec['sample'].topology}\n({rec['style']})", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
#@title 5. Load unsloth/gemma-4-31B-it (Vision Frozen, Language LoRA Spec-Pinned)
import torch

try:
    from unsloth import FastVisionModel as LoaderClass
except ImportError:
    from unsloth import FastModel as LoaderClass

model, tokenizer = LoaderClass.from_pretrained(
    model_name=SPEC["base_model"]["unsloth_id"],
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

try:
    model = LoaderClass.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=SPEC["lora"]["r"],
        lora_alpha=SPEC["lora"]["lora_alpha"],
        lora_dropout=SPEC["lora"]["lora_dropout"],
        bias=SPEC["lora"]["bias"],
        use_gradient_checkpointing="unsloth",
    )
except TypeError:
    model = LoaderClass.get_peft_model(
        model,
        r=SPEC["lora"]["r"],
        target_modules=SPEC["lora"]["target_modules"],
        lora_alpha=SPEC["lora"]["lora_alpha"],
        lora_dropout=SPEC["lora"]["lora_dropout"],
        bias=SPEC["lora"]["bias"],
        use_gradient_checkpointing="unsloth",
    )

model.print_trainable_parameters()


In [ ]:
#@title 6. Multimodal SFT on (Napkin Sketch -> Verified MutinyBIM Script)
from trl import SFTTrainer, SFTConfig

SYS_TEXT = (
    SYSTEM_PROMPT_TEMPLATE
    + "\n## MutinyBIM Script Kit Reference\n"
    + REF_MD[:4000]
)

def build_conversation(rec):
    pil_img = Image.open(rec["image_path"]).convert("RGB")
    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYS_TEXT}]},
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": pil_img},
                    {"type": "text", "text": rec["sample"].user_prompt},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": f"```ts\n{rec['sample'].ts_source.strip()}\n```"}],
            },
        ]
    }

converted_train = [build_conversation(r) for r in train_records]

try:
    from unsloth.trainer import UnslothVisionDataCollator
    data_collator = UnslothVisionDataCollator(model, tokenizer)
except Exception:
    data_collator = None

LoaderClass.for_training(model)
sft_args = SFTConfig(
    output_dir=str(OUT_DIR / "sft"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=MAX_SFT_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=4096,
    save_strategy="no",
    report_to="none",
)

trainer_kwargs = dict(
    model=model,
    tokenizer=tokenizer,
    train_dataset=converted_train,
    args=sft_args,
)
if data_collator is not None:
    trainer_kwargs["data_collator"] = data_collator

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()

final_dir = OUT_DIR / "final_adapter"
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print("Saved spec-compatible vision_floorplan LoRA to:", final_dir)


In [ ]:
#@title 7. Held-Out Eval Gallery (Napkin Sketch | Predicted BIM Plan | Ground Truth) + Hub Push
from verifier.verify_client import extract_typescript

LoaderClass.for_inference(model)
gen_cfg = SPEC["base_model"]["generation_defaults"]
eval_rows = []

for idx, rec in enumerate(eval_records):
    sample = rec["sample"]
    pil_img = Image.open(rec["image_path"]).convert("RGB")
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYS_TEXT}]},
        {"role": "user", "content": [{"type": "image", "image": pil_img}, {"type": "text", "text": sample.user_prompt}]},
    ]
    try:
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text=[prompt_text], images=[pil_img], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=1536,
            temperature=0.4,
            top_p=gen_cfg["top_p"],
            top_k=gen_cfg["top_k"],
            do_sample=True,
        )
    comp = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    pred_ts = extract_typescript(comp)
    pred_bundle = client.export_bundle(pred_ts, project_name="Sketch Eval", include_ifc=False, include_pdf=False)
    gt_bundle = client.export_bundle(sample.ts_source, project_name="Ground Truth", include_ifc=False, include_pdf=False)

    pred_rooms = int(pred_bundle.verify.kinds.get("space", 0))
    gt_rooms = int(sample.expected_kinds.get("space", 0))
    eval_rows.append({
        "sample_id": sample.sample_id,
        "ok": pred_bundle.verify.ok,
        "reward": pred_bundle.verify.reward,
        "clashes": int(pred_bundle.verify.lint_summary.get("errors", 0)),
        "room_exact": pred_rooms == gt_rooms,
        "sketch_path": rec["image_path"],
        "pred_svg": pred_bundle.sheets[0]["svg"] if pred_bundle.sheets else "",
        "gt_svg": gt_bundle.sheets[0]["svg"] if gt_bundle.sheets else "",
    })

n = max(len(eval_rows), 1)
scorecard = {
    "eval_domain": "held_out_parametric_generator",
    "pass_rate": round(sum(1 for r in eval_rows if r["ok"]) / n, 4),
    "zero_clash_rate": round(sum(1 for r in eval_rows if r["clashes"] == 0) / n, 4),
    "mean_reward": round(sum(r["reward"] for r in eval_rows) / n, 4),
    "room_count_exact_match": round(sum(1 for r in eval_rows if r["room_exact"]) / n, 4),
    "benchmark_tasks": len(eval_rows),
}
(final_dir / "eval_scorecard.json").write_text(json.dumps(scorecard, indent=2) + "\n")
print("Held-Out Synthetic Eval Scorecard (measures generator topologies, not real hand-drawn napkins):", json.dumps(scorecard, indent=2))

# Optional: evaluate real hand-drawn napkin sketches if placed in REAL_NAPKIN_DIR
real_dir = Path(REAL_NAPKIN_DIR)
if real_dir.exists():
    real_imgs = sorted([p for p in real_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}])
    if real_imgs:
        print(f"Running eval on {len(real_imgs)} real napkin sketches in {real_dir}...")
        for img_file in real_imgs:
            pil_img = Image.open(img_file).convert("RGB")
            msgs = [
                {"role": "system", "content": [{"type": "text", "text": SYS_TEXT}]},
                {"role": "user", "content": [{"type": "image", "image": pil_img}, {"type": "text", "text": "Convert this hand-drawn architectural floor plan sketch into a complete MutinyBIM TypeScript script with walls, doors, windows, rooms, ceilings, and a plan sheet."}]},
            ]
            try:
                p_txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
            except TypeError:
                p_txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = tokenizer(text=[p_txt], images=[pil_img], return_tensors="pt").to(model.device)
            with torch.no_grad():
                o_ids = model.generate(**inp, max_new_tokens=1536, temperature=0.4, top_p=gen_cfg["top_p"], top_k=gen_cfg["top_k"], do_sample=True)
            comp = tokenizer.decode(o_ids[0][inp["input_ids"].shape[1]:], skip_special_tokens=False)
            rb = client.export_bundle(extract_typescript(comp), project_name=f"Real Napkin {img_file.stem}", include_ifc=False, include_pdf=False)
            print(f"  - {img_file.name}: ok={rb.verify.ok}, reward={rb.verify.reward:.2f}, kinds={rb.verify.kinds}")

registry_entry = {
    "id": "community-vision-floorplan-v1",
    "trade": "vision_floorplan",
    "hf_repo": HF_OUTPUT_REPO,
    "revision": "main",
    "contributor": "5hyycjb77y",
    "method": "sft",
    "default_weight": 0.75,
    "scorecard": scorecard,
}
print("\nRegistry PR snippet:\n", json.dumps(registry_entry, indent=2))

# Render visual comparison for the first 3 eval samples
show_rows = eval_rows[:3]
if show_rows:
    fig, axes = plt.subplots(len(show_rows), 3, figsize=(14, 4.5 * len(show_rows)))
    if len(show_rows) == 1:
        axes = [axes]
    for row_idx, r in enumerate(show_rows):
        axes[row_idx][0].imshow(Image.open(r["sketch_path"]))
        axes[row_idx][0].set_title(f"Input Sketch ({r['sample_id']})")
        if r["pred_svg"]:
            p_png = DATA_DIR / f"eval_pred_{row_idx}.png"
            svg_to_png(r["pred_svg"], p_png, output_width=640)
            axes[row_idx][1].imshow(Image.open(p_png))
        axes[row_idx][1].set_title(f"Predicted BIM Plan (ok={r['ok']}, r={r['reward']:.2f})")
        if r["gt_svg"]:
            g_png = DATA_DIR / f"eval_gt_{row_idx}.png"
            svg_to_png(r["gt_svg"], g_png, output_width=640)
            axes[row_idx][2].imshow(Image.open(g_png))
        axes[row_idx][2].set_title("Ground Truth BIM Plan")
        for c in range(3):
            axes[row_idx][c].axis("off")
    plt.tight_layout()
    plt.show()

if PUSH_TO_HUB:
    from huggingface_hub import HfApi, login
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    api = HfApi()
    api.create_repo(HF_OUTPUT_REPO, exist_ok=True)
    api.upload_folder(folder_path=str(final_dir), repo_id=HF_OUTPUT_REPO)
    print(f"Pushed vision_floorplan LoRA to https://huggingface.co/{HF_OUTPUT_REPO}")
